# Day 7 — End-to-End Preprocessing Pipeline
**Week 3 · Feature Engineering & Data Prep · AI Engineer Lab**

Wire everything from this week into a production-grade scikit-learn Pipeline. This notebook covers:
- Pipeline basics: sequential step chaining
- ColumnTransformer: different treatment per column type
- Custom FunctionTransformer for domain logic
- Safe cross-validation with Pipeline
- GridSearchCV over preprocessing hyperparameters
- Serialising and loading the pipeline for deployment
- Full end-to-end example on real estate data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    StandardScaler, RobustScaler, MinMaxScaler,
    OneHotEncoder, OrdinalEncoder, PowerTransformer, FunctionTransformer
)
from sklearn.feature_selection import SelectFromModel
from sklearn.decomposition import PCA
from sklearn.model_selection import (
    train_test_split, cross_val_score,
    GridSearchCV, cross_validate
)
from sklearn.linear_model import Ridge, LogisticRegression, Lasso
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import mean_squared_error, r2_score, classification_report
from sklearn.datasets import fetch_california_housing

np.random.seed(42)
print('All libraries loaded.')

## 1. Load and Enrich the Dataset

In [ ]:
data = fetch_california_housing(as_frame=True)
df = data.frame.copy()

# Add categorical features (simulated)
np.random.seed(42)
n = len(df)
df['ocean_proximity'] = np.random.choice(
    ['INLAND', 'NEAR_OCEAN', 'NEAR_BAY', '<1H_OCEAN', 'ISLAND'],
    size=n, p=[0.35, 0.25, 0.20, 0.15, 0.05]
)
df['property_age_band'] = pd.cut(
    df['HouseAge'], bins=[0, 10, 20, 30, 50, 100],
    labels=['new', 'recent', 'mature', 'old', 'vintage']
).astype(str)

# Inject missing values
df.loc[np.random.choice(n, int(n * 0.06), replace=False), 'MedInc'] = np.nan
df.loc[np.random.choice(n, int(n * 0.10), replace=False), 'AveRooms'] = np.nan
df.loc[np.random.choice(n, int(n * 0.04), replace=False), 'ocean_proximity'] = np.nan

print('Dataset shape:', df.shape)
print('\nColumn types:')
print(df.dtypes)
print('\nMissing values:')
print(df.isnull().sum())

## 2. Define Feature Groups

In [ ]:
X = df.drop(columns=['MedHouseVal'])
y = df['MedHouseVal']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

# Identify numeric and categorical columns
numeric_features = X.select_dtypes(include=['number']).columns.tolist()
categorical_nominal = ['ocean_proximity']
categorical_ordinal = ['property_age_band']

print(f'\nNumeric features ({len(numeric_features)}): {numeric_features}')
print(f'Categorical nominal: {categorical_nominal}')
print(f'Categorical ordinal: {categorical_ordinal}')

## 3. Build Sub-Pipelines per Column Type

In [ ]:
# ── Numeric pipeline: impute → fix skewness → scale
numeric_pipeline = Pipeline(steps=[
    ('imputer',   SimpleImputer(strategy='median', add_indicator=True)),
    ('transform', PowerTransformer(method='yeo-johnson')),
    ('scaler',    StandardScaler()),
])

# ── Nominal categorical pipeline: impute → OHE
nominal_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False)),
])

# ── Ordinal categorical pipeline: impute → ordinal encode
ordinal_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(
        categories=[['new', 'recent', 'mature', 'old', 'vintage']],
        handle_unknown='use_encoded_value',
        unknown_value=-1
    )),
])

print('Sub-pipelines defined.')
print('numeric_pipeline: impute(median+indicator) → PowerTransform → StandardScale')
print('nominal_pipeline: impute(mode) → OneHotEncode(drop=first)')
print('ordinal_pipeline: impute(mode) → OrdinalEncode(ordered)')

## 4. ColumnTransformer — Combine All Sub-Pipelines

In [ ]:
preprocessor = ColumnTransformer(transformers=[
    ('num',     numeric_pipeline,  numeric_features),
    ('nominal', nominal_pipeline,  categorical_nominal),
    ('ordinal', ordinal_pipeline,  categorical_ordinal),
], remainder='drop')  # drop any unlisted columns

# Test: fit preprocessor alone to inspect output shape
X_train_prep = preprocessor.fit_transform(X_train, y_train)
X_test_prep  = preprocessor.transform(X_test)

print(f'Input shape:  {X_train.shape}')
print(f'Output shape: {X_train_prep.shape}')
print(f'  Numeric features: {len(numeric_features)} numeric + {len(numeric_features)} indicators')
print(f'  Nominal OHE: {X_train_prep.shape[1] - len(numeric_features)*2 - 1} columns')
print(f'  Ordinal: 1 column')

## 5. Full Pipeline: Preprocessor → Model

In [ ]:
# Full pipeline: preprocessor + model
full_pipeline_ridge = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', Ridge(alpha=1.0)),
])

full_pipeline_rf = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', GradientBoostingClassifier()),  # placeholder for RF
])

# Fit and evaluate Ridge
full_pipeline_ridge.fit(X_train, y_train)
y_pred = full_pipeline_ridge.predict(X_test)
rmse = mean_squared_error(y_test, y_pred, squared=False)
r2 = r2_score(y_test, y_pred)
print(f'Ridge Pipeline — RMSE: {rmse:.4f}, R²: {r2:.4f}')

# Compare to naive baseline (predict mean)
baseline_rmse = mean_squared_error(y_test, np.full_like(y_test, y_train.mean()), squared=False)
print(f'Baseline (predict mean) — RMSE: {baseline_rmse:.4f}')
print(f'Improvement: {(1 - rmse/baseline_rmse)*100:.1f}%')

## 6. Safe Cross-Validation — Everything Inside the Fold

In [ ]:
# CORRECT: pass the full pipeline to cross_val_score
# Preprocessing is fitted fresh on each training fold
cv_results = cross_validate(
    full_pipeline_ridge, X, y,
    cv=5,
    scoring={'rmse': 'neg_root_mean_squared_error', 'r2': 'r2'},
    return_train_score=True
)

print('=== 5-Fold Cross-Validation Results (Ridge + Full Preprocessor) ===')
print(f'CV RMSE: {-cv_results["test_rmse"].mean():.4f} ± {cv_results["test_rmse"].std():.4f}')
print(f'CV R²:   {cv_results["test_r2"].mean():.4f} ± {cv_results["test_r2"].std():.4f}')
print()

# Show that train vs test gap is small (no overfitting)
print(f'Train RMSE: {-cv_results["train_rmse"].mean():.4f}')
print(f'Test  RMSE: {-cv_results["test_rmse"].mean():.4f}')
print('Small gap → model generalises well, no data leakage.')

## 7. GridSearchCV Over Pipeline Parameters

In [ ]:
# Search over imputation strategy, scaler, and model regularisation
param_grid = {
    'preprocessor__num__imputer__strategy': ['mean', 'median'],
    'preprocessor__num__scaler': [StandardScaler(), RobustScaler()],
    'model__alpha': [0.1, 1.0, 10.0, 100.0],
}

gs = GridSearchCV(
    full_pipeline_ridge,
    param_grid,
    cv=3,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    verbose=1,
    refit=True
)

gs.fit(X_train, y_train)
print(f'\nBest params: {gs.best_params_}')
print(f'Best CV RMSE: {-gs.best_score_:.4f}')

# Evaluate best pipeline on test set
y_pred_best = gs.best_estimator_.predict(X_test)
final_rmse = mean_squared_error(y_test, y_pred_best, squared=False)
print(f'Test RMSE with best pipeline: {final_rmse:.4f}')

## 8. Custom FunctionTransformer — Domain Feature Engineering

In [ ]:
def add_domain_features(X):
    """Add domain-specific engineered features."""
    df = pd.DataFrame(X, columns=numeric_features)
    df['rooms_per_person'] = df['AveRooms'] / df['AveOccup'].clip(lower=0.1)
    df['bedrooms_ratio']   = df['AveBedrms'] / df['AveRooms'].clip(lower=0.1)
    df['income_location']  = df['MedInc'] * (1 / (1 + np.abs(df['Latitude'] - 35)))
    return df.values

# Wrap as sklearn transformer
domain_transformer = FunctionTransformer(add_domain_features, validate=False)

# Updated numeric pipeline with domain features
extended_numeric_pipeline = Pipeline(steps=[
    ('imputer',   SimpleImputer(strategy='median')),
    ('domain',    FunctionTransformer(add_domain_features, validate=False)),
    ('scaler',    StandardScaler()),
])

# Test it
X_num_only = X_train[numeric_features]
X_extended = extended_numeric_pipeline.fit_transform(X_num_only)
print(f'Numeric features before domain engineering: {len(numeric_features)}')
print(f'After domain engineering: {X_extended.shape[1]} features')

## 9. Serialize and Load Pipeline for Deployment

In [ ]:
import os

# Save the best fitted pipeline
pipeline_path = 'preprocessing_pipeline_w3.pkl'
joblib.dump(gs.best_estimator_, pipeline_path)
file_size_kb = os.path.getsize(pipeline_path) / 1024
print(f'Pipeline saved to: {pipeline_path} ({file_size_kb:.1f} KB)')

# Load and use — this is what production inference looks like
loaded_pipeline = joblib.load(pipeline_path)

# Simulate new raw data arriving at inference time
new_data = X_test.head(5).copy()
new_data.loc[new_data.index[0], 'MedInc'] = np.nan    # simulate missing value
new_data.loc[new_data.index[1], 'ocean_proximity'] = np.nan

# Pipeline handles everything automatically
predictions = loaded_pipeline.predict(new_data)
print(f'\nPredictions on 5 new (raw) samples: {predictions.round(4)}')
print('Pipeline handled missing values, encoding, scaling — all in one call!')

# Clean up
os.remove(pipeline_path)

## 10. Pipeline Inspection — What Did We Build?

In [ ]:
from sklearn import set_config
set_config(display='text')

# Print pipeline structure
print(full_pipeline_ridge)

# Access individual fitted components
prep = full_pipeline_ridge.named_steps['preprocessor']
num_pipe = prep.named_transformers_['num']

print('\nImputer strategy:', num_pipe.named_steps['imputer'].strategy)
print('Scaler type:', type(num_pipe.named_steps['scaler']).__name__)
print('Scaler mean (first 3):', num_pipe.named_steps['scaler'].mean_[:3].round(4))

# OHE categories learned from training data
ohe = prep.named_transformers_['nominal'].named_steps['encoder']
print('\nOHE categories for ocean_proximity:', ohe.categories_[0])
print('OHE output column names:', ohe.get_feature_names_out(['ocean_proximity']))

## 11. Leakage Proof: Manual vs Pipeline Cross-Validation

In [ ]:
from sklearn.model_selection import KFold

# WRONG WAY: preprocess outside CV — leaks test-fold statistics into training
# (Imputer sees the entire dataset including validation fold data)
leaky_imputer = SimpleImputer(strategy='median')
leaky_scaler  = StandardScaler()
X_numeric = X[numeric_features].values
X_leaky = leaky_imputer.fit_transform(X_numeric)      # fits on ALL data
X_leaky = leaky_scaler.fit_transform(X_leaky)          # fits on ALL data

lr = Ridge(alpha=1.0)
leaky_rmse = -cross_val_score(lr, X_leaky, y, cv=5,
                               scoring='neg_root_mean_squared_error').mean()

# CORRECT WAY: preprocessing inside pipeline
correct_pipe = Pipeline([
    ('imp', SimpleImputer(strategy='median')),
    ('sc',  StandardScaler()),
    ('lr',  Ridge(alpha=1.0))
])
correct_rmse = -cross_val_score(correct_pipe, X_numeric, y, cv=5,
                                 scoring='neg_root_mean_squared_error').mean()

print('=== Leakage Demo ===')
print(f'WRONG (preprocess outside CV): RMSE = {leaky_rmse:.4f}  ← optimistically biased')
print(f'CORRECT (pipeline + CV):       RMSE = {correct_rmse:.4f}  ← honest estimate')
print(f'Leakage bias: {leaky_rmse - correct_rmse:.4f} ({(leaky_rmse/correct_rmse - 1)*100:.2f}%)')

## Exercises

1. **Classification pipeline**: Build a full Pipeline for a binary classification task (e.g., predicting if house value > median). Include all preprocessing steps, use GradientBoostingClassifier, and evaluate with AUC.

2. **PCA in pipeline**: Add a PCA step (n_components=0.95) after the preprocessor and before the model. Does it improve or hurt Ridge regression performance? Why?

3. **Multiple models comparison**: Create 3 pipelines (same preprocessor, different models: Ridge, Lasso, RandomForest). Compare them with 5-fold CV. Which preprocessor works best for each?

4. **Custom transformer class**: Create a full sklearn-compatible transformer class (`fit`, `transform`, `get_feature_names_out`) that adds interaction features (e.g., `income * rooms`). Integrate it into the pipeline.

In [ ]:
# ── Exercise 4 Solution: Custom transformer class ──
from sklearn.base import BaseEstimator, TransformerMixin

class InteractionFeatures(BaseEstimator, TransformerMixin):
    """Add pairwise interaction features for specified column pairs."""
    
    def __init__(self, pairs):
        self.pairs = pairs  # list of (col_i, col_j) tuples
    
    def fit(self, X, y=None):
        return self  # stateless, no fitting needed
    
    def transform(self, X):
        X_arr = np.array(X)
        interactions = [X_arr[:, i] * X_arr[:, j] for i, j in self.pairs]
        return np.hstack([X_arr] + [v.reshape(-1, 1) for v in interactions])
    
    def get_feature_names_out(self, input_features=None):
        base = list(input_features) if input_features else [f'x{i}' for i in range(self.n_features_in_)]
        interaction_names = [f'{base[i]}_x_{base[j]}' for i, j in self.pairs]
        return np.array(base + interaction_names)

# Use in pipeline
X_num_demo = X_train[numeric_features].fillna(0).values
interaction_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('interact', InteractionFeatures(pairs=[(0, 1), (0, 2), (1, 3)])),  # pairs of feature indices
    ('scaler', StandardScaler()),
])

X_with_interactions = interaction_pipe.fit_transform(X_num_demo)
print(f'Features before interactions: {len(numeric_features)}')
print(f'Features after adding 3 interactions: {X_with_interactions.shape[1]}')